# GPS with time bias

## GPS Mathematical Model
Your mobile phone is a GPS receiver. Assume that your phone receive signal from satellites. Based on that, you can calculate your position

Let:
- Your phone position: $ (x, y, z) $
- Time-bias: $ b $
- Given satellite position and the time on the satellite clock when signal is sent: $ (x_i, y_i, z_i) $ and $ t_i $
- Given time on your phone when it receive signal: $t_r$
- Given speed of light: $c = 300000000 $  m/s

<!-- - Receiver clock bias: $ b $ -->

Then, for each satellite:

$$
c \cdot (t_r - t_i)=\rho_i = \sqrt{(x - x_i)^2 + (y - y_i)^2 + (z - z_i)^2} + c \cdot b
$$


Where:
- $ \rho_i $: pseudo range
<!-- - $$ c \cdot b $$: clock bias in meters -->
---


In [1]:
# import required libraries
import numpy as np

# satellite global positions
sat_pos = np.array([
   [2.467090943105902,   0.156574709947754,  -0.971148079074766],
  [-0.319502626790938,  -1.489529356829127,   2.175638370561362],
   [2.108149976088853,   1.250310757401082,   1.023038924520812],
   [1.866521392093421,   0.086255224515095,   1.887548643596368],
   [1.659419428111197,  -0.744299998415712,   1.935594644238942],
   [1.530252094230724,  -1.705237261413833,   1.343382433720694],
   [0.650232928798093,   1.404746463854678,   2.158255421268630],
  [-0.751093932597779,   1.590584973862852,   1.989994798530266],
   [1.323440643586666,   2.190632726566670,   0.709799318849342],
]) * 1e7

# according time on the atomic clock when these satellites send signal
time_send = np.array([
   0.916175267449982,
   0.919565688561347,
   0.928685046990766,
   0.932570145119642,
   0.930720774042633,
   0.924635031123122,
   0.927886987479063,
   0.918478177382462,
   0.922446321701693,
])

# time on your phone when it receive signal
time_receive = 1.0

# light speed
c = 3e8

## Solving with Least Squares

This is a **nonlinear system**

For the nonlinear system, a good way to solve it is using numerical methods.

In this example, we are going solve it using least squares (from scipy library). Document for scipy least square: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html

This function tries to reduce the residual to 0. From the GPS function, the residual is:
$$
\sqrt{(x - x_i)^2 + (y - y_i)^2 + (z - z_i)^2} + c \cdot b - c \cdot (t_r - t_i) = 0
$$

We want to 

$$
\min_{x,y,z,b} \sum_{i=1}^{N} (\sqrt{(x - x_i)^2 + (y - y_i)^2 + (z - z_i)^2} + c \cdot b - c \cdot (t_r - t_i))^2
$$

This finds the best estimate of position and clock bias.

---

In [2]:
# import lesate_squares from scipy
from scipy.optimize import least_squares

# set up target to optimize / residuals
def target(state, sat_pos, time_send, time_receive):
    x, y, z, b = state

    # predicted pseudo-range for each satellite
    rho_pred = np.sqrt(
        (x - sat_pos[:, 0])**2 +
        (y - sat_pos[:, 1])**2 +
        (z - sat_pos[:, 2])**2
    ) + c * b

    # measured pseudo-range from satellite transmission time
    rho_meas = c * (time_receive - time_send)

    # return residuals (one per satellite)
    return rho_pred - rho_meas

# initial guess
x0 = np.array([0, 0, 0, 0], dtype=float)

# run least squares: minimize sum of squared residuals
result = least_squares(target, x0, args=(sat_pos, time_send, time_receive))

x_est, y_est, z_est, b = result.x
print("Estimated receiver position [m]:")
print("X: ", x_est)
print("Y: ", y_est)
print("Z: ", z_est)
print("Time bias: ", b)

Estimated receiver position [m]:
X:  4148010.5075931423
Y:  612607.7612412696
Z:  4790067.902819938
Time bias:  -4.64693279398092e-09


## From ECEF to LLA (Geodetic Coordinates)

Previous positions are computed in the **ECEF frame** (Earth-Centered, Earth-Fixed):


To convert ECEF to LLA, we can use the online tools (https://www.convertecef.com/)

Reference localtion: https://maps.app.goo.gl/712ak3hJRmvxZzrGA